In [1]:
# ================================================================
#  LCCN-R21a - PHONETIC GUARD DECOMPOSITION: ARMS, ACCOUNTING,
#  CLASSICAL MACRO-F1  [CPU, ~40 min]
#  Answers IJIES Reviewer 2, point 1 (association vs mechanism).
#  soundex_guard / indicsoundex_guard apply all three vetoes; NB1 logged
#  that the LENGTH veto blocked most merges (2,230 of 3,280 for Soundex).
#  This notebook isolates each veto: six leave-one-in arms built with
#  the certified elected() logic of pipeline v5.3 and ONE veto each:
#    soundex_emotion   soundex_length   soundex_negation
#    indicsoundex_emotion   indicsoundex_length   indicsoundex_negation
#  plus the four certified references rebuilt here for a same-run check
#  (soundex, soundex_guard, indicsoundex, indicsoundex_guard).
#  Accounting per arm (training split): forms rewritten, merged pairs,
#  X(f), merge precision; tokens changed over the full corpus.
#  Then LR, SVM, Naive Bayes, Random Forest, XGBoost (trainer v1.4).
#  Writes guard_texts.csv for the GPU notebook LCCN-R21b.
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml
#             (output, trainer.py), final-emotion-dataset, gold-ds.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library xgboost')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library xgboost
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 4.9 MB/s eta 0:00:00
python 3.12.13


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - SINGLE-VETO GUARDS AND THE TEN PHONETIC ARMS
#  elected() is copied from build_arms() in pipeline v5.3 with the veto
#  passed in; the bucket election over TRAIN_FREQ is unchanged.
# ================================================================
from collections import defaultdict, Counter
from tqdm.auto import tqdm

def veto_only(kind, theta=3):
    def veto(w, c, prev=None):
        if kind == "emotion":
            a, b = LEX.emotion_in_context(w, prev), LEX.emotion_in_context(c, prev)
            if a and b and len(a) == 1 and len(b) == 1 and a != b: return "emotion"
        elif kind == "negation":
            if (w in pipeline.NEG) != (c in pipeline.NEG): return "negation"
        elif kind == "length":
            if abs(len(w) - len(c)) > theta: return "length"
        elif kind == "all":
            return make_veto(F2E, LEX)(w, c, prev)
        return None
    return veto

def elected(keyfn, veto=None):
    b = defaultdict(Counter)
    for w, n in TRAIN_FREQ.items():
        kk = keyfn(w)
        if kk: b[kk][w] += n
    m = {w: c.most_common(1)[0][0] for kk, c in b.items() for w in c}
    vet = Counter()
    if veto is not None:
        out = {}
        for w, c in m.items():
            g = veto(w, c) if w != c else None
            if g: out[w] = w; vet[g] += 1
            else: out[w] = c
        m = out
    def fn(text):
        toks, tags = process_comment(text)
        return " ".join(w if k in PASSTHROUGH else m.get(w.lower(), w) for w, k in zip(toks, tags))
    return fn, m, dict(vet)

ARMS_DEF = [("soundex", soundex, None), ("soundex_guard", soundex, "all"),
            ("soundex_emotion", soundex, "emotion"), ("soundex_length", soundex, "length"),
            ("soundex_negation", soundex, "negation"),
            ("indicsoundex", indic_soundex, None), ("indicsoundex_guard", indic_soundex, "all"),
            ("indicsoundex_emotion", indic_soundex, "emotion"), ("indicsoundex_length", indic_soundex, "length"),
            ("indicsoundex_negation", indic_soundex, "negation")]

single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
train_mask = df[ID].isin(SPLIT["train"]).values
train_vocab = set(TRAIN_FREQ)

rows = []
for name, keyfn, vk in ARMS_DEF:
    fn, m, vet = elected(keyfn, veto_only(vk) if vk else None)
    df[f"text_{name}"] = df[TXT].map(fn)
    # accounting on the training vocabulary mapping m (exactly the Table 5 rule)
    rewritten = {w for w, c in m.items() if c != w and w in train_vocab}
    buckets = defaultdict(set)
    for w, c in m.items():
        if w in train_vocab: buckets[c].add(w)
    pairs = lab = same = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2: continue
        pairs += n*(n-1)//2
        ce = Counter(single[f] for f in forms if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x = lab - same; prec = 1 - x/lab if lab else float("nan")
    ch = tot = 0
    for a, b_ in zip(df["text_raw"], df[f"text_{name}"]):
        ra, rb = a.lower().split(), b_.lower().split(); assert len(ra) == len(rb), name
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    row = dict(arm=name, veto=vk or "none", tokens_changed_full_pct=round(100*ch/tot, 2),
               forms_rewritten_train=len(rewritten), merged_pairs_train=pairs,
               labelled_pairs_train=lab, x_cross_train=x, merge_precision_train=round(prec, 4),
               vetoed_total=sum(vet.values()), **{f"vetoed_{k}": v for k, v in vet.items()})
    rows.append(row)
    print(f"{name:<24} tok {row['tokens_changed_full_pct']:5.2f}%  forms_rew {len(rewritten):>6,}  "
          f"pairs {pairs:>7,}  X {x:>5,}  prec {prec:.4f}  vetoed {vet}")
acc = pd.DataFrame(rows); acc.to_csv("/kaggle/working/guard_accounting.csv", index=False)
print("\n", acc.to_string(index=False))
df[[ID] + [f"text_{n}" for n, *_ in ARMS_DEF]].to_csv("/kaggle/working/guard_texts.csv", index=False, encoding="utf-8-sig")
print("\nCONSISTENCY: soundex and soundex_guard must reproduce Table 5 (merged pairs 15,993 / 15,431;"
      " X 1,064 / 500); indicsoundex / indicsoundex_guard (16,133 / 15,491; X 1,104 / 517).")


soundex                  tok 27.25%  forms_rew 23,158  pairs 342,835  X 8,534  prec 0.4403  vetoed {}
soundex_guard            tok 24.39%  forms_rew 19,878  pairs 273,543  X 4,394  prec 0.5445  vetoed {'length': 2230, 'emotion': 845, 'negation': 205}
soundex_emotion          tok 25.98%  forms_rew 22,313  pairs 322,566  X 4,754  prec 0.5513  vetoed {'emotion': 845}
soundex_length           tok 26.25%  forms_rew 20,829  pairs 295,314  X 7,623  prec 0.4375  vetoed {'length': 2329}
soundex_negation         tok 26.58%  forms_rew 22,948  pairs 337,673  X 8,499  prec 0.4407  vetoed {'negation': 210}
indicsoundex             tok 27.15%  forms_rew 23,184  pairs 338,498  X 8,449  prec 0.4414  vetoed {}
indicsoundex_guard       tok 24.08%  forms_rew 19,858  pairs 269,244  X 4,195  prec 0.5516  vetoed {'length': 2252, 'emotion': 873, 'negation': 201}
indicsoundex_emotion     tok 25.66%  forms_rew 22,311  pairs 317,403  X 4,529  prec 0.5589  vetoed {'emotion': 873}
indicsoundex_length      tok 26.1

In [5]:
# ================================================================
#  CELL 4 - CLASSICAL MACRO-F1 (trainer v1.4, byte-identical from NB2)
# ================================================================
import shutil, glob, os, sys, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py"); sys.modules.pop("trainer", None)
import trainer; importlib.reload(trainer)
from trainer import *
assert trainer.TRAINER_VERSION == "1.4"; trainer.fingerprint()
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
NEW = [f"text_{n}" for n, *_ in ARMS_DEF]
lut = {c: dict(zip(df[ID].astype(str), df[c])) for c in NEW}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in NEW:
        f[c] = ids.map(lut[c]); assert f[c].notna().all(), c
ARMS = ["raw"] + [n for n, *_ in ARMS_DEF]
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM", "Naive Bayes", "Random Forest", "XGBoost"],
               tag="lccn_r21_guard_classical", resume=False)
print("\n=== classical macro-F1 ===")
print(res.to_string(index=False))
print("\nCHECK: soundex / soundex_guard / indicsoundex / indicsoundex_guard columns must equal Table 3.")
print("DONE - guard_accounting.csv, guard_texts.csv, results_lccn_r21_guard_classical.csv written.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / raw ===
  -> macro-F1 0.8033

=== Logistic Regression / soundex ===
  -> macro-F1 0.7760

=== Logistic Reg